In [ ]:
import taichi as ti
from topologies import square_torus
ti.init(arch=ti.gpu)

In [ ]:



neuron_count = 10000
mps = ti.field(dtype=ti.float32, shape=neuron_count)
# weights = WeightMatrix(square_torus(100))
inputs = ti.field(dtype=ti.float32, shape=neuron_count)
spiking_neurons = ti.field(dtype=ti.int64, shape=neuron_count)
running = True
SPIKE_THRESHOLD = 1





In [ ]:
@ti.kernel
def get_spiking_neurons(result: ti.types.vector(1, ti.int64)):
    i = 0
    for n in range(neuron_count):
        if mps[n] > SPIKE_THRESHOLD:
            result[i] = n
        i+=1
        
@ti.kernel
def spike_neurons(neurons: ti.template()):
    for n in neurons:
        for j in ti.static(range(4)):
            child_idx = weights.get_neighbor(n, j)
            inputs[child_idx] += weights.get_weight(n, child_idx)

@ti.kernel
def update_mps():
    for n in range(neuron_count):
        mps[n] += inputs[n]

mps.fill(0.1)

while running:
    update_mps()
    inputs.fill(0.0)
    spiking_neurons.fill(0.0)

    get_spiking_neurons(spiking_neurons)
    spike_neurons(spiking_neurons)





